# create models from ASVset genome objects

In [1]:
from load_genomes_to_modelseedpy import GenomeLoader
from cobra.io import write_sbml_model
# from glob import glob
from tqdm import tqdm
import os

loader = GenomeLoader()
# genome_ids = loader.list_available_genomes()#[:10]

def write_model(gID):
    model = loader.build_model(gID, printing=False)
    write_sbml_model(model, f'models/{gID}.xml')

# for genome_id in tqdm(genome_ids):
#     model_path = f'models/{genome_id}.xml'
#     if os.path.exists(model_path):  continue
#     try:
#         model = loader.build_model(genome_id, printing=False)
#         write_sbml_model(model, model_path)
#         # print(f"✓ {genome_id}") 
#     except Exception as e:
#         print(f"✗ {genome_id}: {e}")


# parallelize the effort
args = [gID for gID in loader.list_available_genomes() if not os.path.exists(f'models/{gID}.xml')]#[:2]
print(len(args), "will be processed")
parallelize = False
if parallelize:
    from multiprocess import Pool
    from os import cpu_count
    
    cpus = int(cpu_count()/8)
    print(f"{cpus} cores are being used.  The first argument is {args[0]}")
    pool = Pool(cpus)
    outputs = pool.map(write_model, args)
    
    print(f"missing iterativeIDs", missing_iterativeIDs)
    print(f"missing IDs", missing_IDs)
else:
    for arg in args:
        # print(arg)
        write_model(arg)

100%|██████████| 1698/1698 [1:21:00<00:00,  2.86s/it]  


## Gapfill the models in minimal media

## Define the metabolite-centered phenotypes

In [12]:
%run util.py
from json import load
from numpy import mean

# microbiome_path = "/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/digestor"
microbiome_path = "../MicrobiomeNotebooks"
metabolites = load(open(f"{microbiome_path}/digestor/metabolites.json", 'r'))
metaboliteIDs = list(metabolites.keys())

media_nar = 207617
metabolites = util.msrecon.get_media(f"{media_nar}/Wolfe").mediacompounds
auxo_media = util.msrecon.get_media(f"{media_nar}/AuxoMedia")
anaerobicPyruvate = util.msrecon.get_media(f"{media_nar}/PyruateMinimalAnaerobic")
# metabolites
metaboliteIDs = [cpd['id'] for cpd in metabolites] + ["cpd00011", "cpd11640", "cpd01024"]  # experimental media + CO2,H2,CH4

#TODO:  Why is growth only performed for the non-base media and uptake/excretion only performed for the base media?
uptake_phenoset = util.create_phenotypeset_from_compounds(
    metaboliteIDs, base_media=auxo_media, base_uptake=0, base_excretion=1000, global_atom_limits={}, type="uptake")
excretion_phenoset = util.create_phenotypeset_from_compounds(
    metaboliteIDs, base_media=auxo_media, base_uptake=0, base_excretion=1000, global_atom_limits={}, type="excretion")
agrowth_phenoset = util.create_phenotypeset_from_compounds(
    metaboliteIDs, base_media=anaerobicPyruvate, base_uptake=0, base_excretion=1000, global_atom_limits={}, type="agrowth")
# growth_phenoset = util.create_phenotypeset_from_compounds(
#     metabolites, base_media=gmm_base_media, base_uptake=0, base_excretion=1000, global_atom_limits={}, type="growth")
phenotypes = {"uptake": uptake_phenoset, "excretion": excretion_phenoset, "agrowth": agrowth_phenoset}

python version 3.12.3


['/home/afreiburger/repos/cobrakbase',
 '/home/afreiburger/repos/KBBaseModules',
 '/home/afreiburger/repos/chenry_utility_module/lib',
 '/home/afreiburger/repos/cb_annotation_ontology_api/lib',
 '/home/afreiburger/repos/KB-ModelSEEDReconstruction/lib',
 '/home/afreiburger/repos/MergeMetabolicAnnotations/lib',
 '/home/afreiburger/repos/codiffusion_bioreactor',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/home/afreiburger/repos/.venv/lib/python3.12/site-packages',
 '/home/afreiburger/repos/.venv/lib/python3.12/site-packages/libsbml',
 '/home/afreiburger/repos']

KBBaseModules 0.0.1
cobrakbase 0.3.1
Output files printed to:/home/afreiburger/repos/codiffusion_bioreactor/Sludge/nboutput when using KBDevUtils.output_dir


# create community models from the member models and abundances

In [1]:
from cobra.io import read_sbml_model, write_sbml_model
from pandas import read_csv, Series
from mscommunity import build_from_species_models
from tqdm import tqdm
from json import load
import os

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))
ASV_sets = load(open("modeling_files/ASV_sets.json", 'r'))
redunant_asvs = {v:k for k,vs in ASV_sets.items() for v in vs}
abundances = load(open("modeling_files/ASVset_abundances.json", 'r'))
abundances = {sample: {iterativeIDs.get(ASV): abund for ASV, abund in content.items()}
              for sample, content in abundances.items()}
abundances_new = {sample: {iterativeIDs.get(ASV): {"abundance": abund} for ASV, abund in content.items()}
              for sample, content in abundances.items()}

def create_comm(item):
    sample, abundances = item
    models = []
    for ASVset, abun in abundances.items():
        # ASVset = iterativeIDs[redunant_asvs.get(ASV, ASV)]
        # print(ASVset)
        model_path = f"gapfilled_models/{ASVset}_gf.xml"
        #TODO:  investigate why these ASVs did not have any genomeIDs
        if not os.path.exists(model_path):  # some ASVs have no genomeIDs
            continue
            
        model = read_sbml_model(model_path)
        model.id = ASVset
        models.append(model)
    modelID = f"{sample}_comm"
    modelName = f"{sample}_comm"
    print(len(models))
    comm_model = build_from_species_models(models, modelID, modelName, printing=True) #abundances_new, printing=True)
    write_sbml_model(comm_model, f"comm_models/{modelID}.xml")

# parallelize the effort
args = [item for item in abundances.items() if not os.path.exists(f"comm_models/{item[0]}_comm.xml")]
print(len(args), "will be processed")
parallelize = False
if parallelize:
    from multiprocess import Pool
    from os import cpu_count
    
    cpus = min(len(args), int(cpu_count()/8))
    print(f"{cpus} cores are being used.  The first argument is {args[0]}")
    pool = Pool(cpus)
    outputs = pool.map(create_comm, args)
    
    print(f"missing iterativeIDs", missing_iterativeIDs)
    print(f"missing IDs", missing_IDs)
else:
    for arg in args:
        # print(arg)
        create_comm(arg)
        # break

modelseedpy 0.3.3


/home/afreiburger/repos/MSCommunity/mscommunity/commhelper.py:25: SyntaxWarning: invalid escape sequence '\_'
  return re.sub("(\_\w\d)", "", ID)
/home/afreiburger/repos/MSCommunity/mscommunity/commhelper.py:111: SyntaxWarning: invalid escape sequence '\d'
  if re.search('^(bio)(\d+)$', rxn.id) or "biomass" in rxn.id:


19 will be processed
156
156 models are being processed
1058 metabolites of midas_g_4042.3 are being processed
midas_g_4042.3 biomass metabolite captured cpd11416_c1
1048 reactions of midas_g_4042.3 are being processed
bio1 from midas_g_4042.3 becomes bio2
987 metabolites of midas_g_269.4 are being processed
midas_g_269.4 biomass metabolite captured cpd11416_c2
986 reactions of midas_g_269.4 are being processed
bio1 from midas_g_269.4 becomes bio3
941 metabolites of midas_g_52.2 are being processed
midas_g_52.2 biomass metabolite captured cpd11416_c3
950 reactions of midas_g_52.2 are being processed
bio1 from midas_g_52.2 becomes bio4
796 metabolites of OPB41.1 are being processed
OPB41.1 biomass metabolite captured cpd11416_c4
812 reactions of OPB41.1 are being processed
bio1 from OPB41.1 becomes bio5
787 metabolites of midas_g_531.1 are being processed
midas_g_531.1 biomass metabolite captured cpd11416_c5
784 reactions of midas_g_531.1 are being processed
bio1 from midas_g_531.1 beco

In [15]:
from json import load
abundances = load(open("modeling_files/ASVset_abundances.json", 'r'))
list(list(abundances.values())[0].values())[0]

0.0035438998239284624

## Test that the models are operational

In [1]:
from cobra.io import read_sbml_model
from glob import glob

# for path in glob("models/F34_comm.xml"):
# for path in glob("comm_models/G12_comm.xml"):
    # print(path.split("/")[-1])
model = read_sbml_model("comm_models/P12_comm.xml")
# break

In [2]:
print(model.objective)
display(model.summary())

Maximize
1.0*bio1 - 1.0*bio1_reverse_b18f7


Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00013_e0,EX_cpd00013_e0,514.8,0,0.00%
cpd00023_e0,EX_cpd00023_e0,10,5,5.26%
cpd00027_e0,EX_cpd00027_e0,10,6,6.32%
cpd00030_e0,EX_cpd00030_e0,0.1906,0,0.00%
cpd00033_e0,EX_cpd00033_e0,10,2,2.11%
cpd00034_e0,EX_cpd00034_e0,0.1906,0,0.00%
cpd00035_e0,EX_cpd00035_e0,10,3,3.16%
cpd00039_e0,EX_cpd00039_e0,10,6,6.32%
cpd00041_e0,EX_cpd00041_e0,10,4,4.21%
cpd00051_e0,EX_cpd00051_e0,10,6,6.32%


In [3]:
model.reactions.bio1.reaction

'0.0277777777777778 cpd11416_c1 + 0.0277777777777778 cpd11416_c10 + 0.0277777777777778 cpd11416_c11 + 0.0277777777777778 cpd11416_c12 + 0.0277777777777778 cpd11416_c13 + 0.0277777777777778 cpd11416_c14 + 0.0277777777777778 cpd11416_c15 + 0.0277777777777778 cpd11416_c16 + 0.0277777777777778 cpd11416_c17 + 0.0277777777777778 cpd11416_c18 + 0.0277777777777778 cpd11416_c19 + 0.0277777777777778 cpd11416_c2 + 0.0277777777777778 cpd11416_c20 + 0.0277777777777778 cpd11416_c21 + 0.0277777777777778 cpd11416_c22 + 0.0277777777777778 cpd11416_c23 + 0.0277777777777778 cpd11416_c24 + 0.0277777777777778 cpd11416_c25 + 0.0277777777777778 cpd11416_c26 + 0.0277777777777778 cpd11416_c27 + 0.0277777777777778 cpd11416_c28 + 0.0277777777777778 cpd11416_c29 + 0.0277777777777778 cpd11416_c3 + 0.0277777777777778 cpd11416_c30 + 0.0277777777777778 cpd11416_c31 + 0.0277777777777778 cpd11416_c32 + 0.0277777777777778 cpd11416_c33 + 0.0277777777777778 cpd11416_c34 + 0.0277777777777778 cpd11416_c35 + 0.02777777777777

In [4]:
bioReactions = []
for rxn in model.reactions:
    if "bio" in rxn.id:
        bioReactions.append(rxn.id)
bioReactions = sorted(bioReactions)
print(len(bioReactions), bioReactions)

37 ['bio1', 'bio10', 'bio11', 'bio12', 'bio13', 'bio14', 'bio15', 'bio16', 'bio17', 'bio18', 'bio19', 'bio2', 'bio20', 'bio21', 'bio22', 'bio23', 'bio24', 'bio25', 'bio26', 'bio27', 'bio28', 'bio29', 'bio3', 'bio30', 'bio31', 'bio32', 'bio33', 'bio34', 'bio35', 'bio36', 'bio37', 'bio4', 'bio5', 'bio6', 'bio7', 'bio8', 'bio9']


In [5]:
bioCompounds = []
for cpd in model.metabolites:
    if "cpd11416" in cpd.id:
        bioCompounds.append(cpd.id)
bioCompounds = sorted(bioCompounds)
print(len(bioCompounds), bioCompounds)

37 ['cpd11416_c0', 'cpd11416_c1', 'cpd11416_c10', 'cpd11416_c11', 'cpd11416_c12', 'cpd11416_c13', 'cpd11416_c14', 'cpd11416_c15', 'cpd11416_c16', 'cpd11416_c17', 'cpd11416_c18', 'cpd11416_c19', 'cpd11416_c2', 'cpd11416_c20', 'cpd11416_c21', 'cpd11416_c22', 'cpd11416_c23', 'cpd11416_c24', 'cpd11416_c25', 'cpd11416_c26', 'cpd11416_c27', 'cpd11416_c28', 'cpd11416_c29', 'cpd11416_c3', 'cpd11416_c30', 'cpd11416_c31', 'cpd11416_c32', 'cpd11416_c33', 'cpd11416_c34', 'cpd11416_c35', 'cpd11416_c36', 'cpd11416_c4', 'cpd11416_c5', 'cpd11416_c6', 'cpd11416_c7', 'cpd11416_c8', 'cpd11416_c9']


# Community modeling

## Load the community models

In [ ]:
from cobra.io import read_sbml_model
from tqdm import tqdm
from glob import glob

models = {}
num_models = 2
for model_path in tqdm(glob("comm_models/*.xml")[:num_models]):
    modelID = model_path.split("/")[-1].replace(".xml", '')
    print("loading", modelID)
    model = read_sbml_model(model_path)
    model.id = modelID
    models[modelID] = model

  0%|          | 0/2 [00:00<?, ?it/s]'' is not a valid SBML 'SId'.


loading 10AB_comm


## Constraints

### Process data

#### convert measurements into MSIDs

In [ ]:
from collections import Counter
from pandas import read_csv, set_option
from json import dump

set_option("display.max_columns", None)

mapping = {"Media Acetate (mol/L)": "cpd00029_in", "Media Propionate (mol/L)": "cpd00141_in",
           "Media Butyrate (mol/L)": "cpd00211_in", "Waste Effluent Acetate (mol/L)": "cpd00029_out",
           "Waste Effluent Propionate (mol/L)": "cpd00141_out", "Waste Effluent Butyrate (mol/L)": "cpd00211_out",
           "Approximate Total Influent C (mol/min) based on Media recipe": "media_carbon_in",
           "Media Total COD (mg/L) Hach": "media_cpd00007",
           "Measured Total Influent C (mol/min)": "carbon_in",
           "Gas Composition (% CH4)": "%cpd01024", "Gas Composition (% H2)": "%cpd11640", "Gas Composition (% CO2)": "%cpd00011",
           "CH4 breakthrough (mol/min)": "cpd01024_out", "H2 breakthrough (mol/min)": "cpd11640_out", "CO2 breakthrough (mol/min)": "cpd00011_out",
           "Accounted C (%) based on MT sensors": "accounted_C"}


data = read_csv("model_inputs/measurements/Summary_interpolated.csv")
data["Media Total COD (mg/L) Hach"] /= 32000
data["carbon_out"] = data["Waste Effluent Dissolved Inorganic Carbon (mol C/L)"] + data["Waste Effluent Dissolved Organic Carbon (mol C/L)"
                        ] + data["Effluent C as measured (mol/min)"]
data.rename(columns=mapping, inplace=True)
data = data[data["Biomass Sample ID"].notna()].set_index("Biomass Sample ID")
# print(Counter(data.columns))
data_df = data.T.to_dict()
dump(data_df, open("model_inputs/measurements/Summer_interpolated_MSDB.json", 'w'), indent=2)
data

In [ ]:
data_df

### loading the media

In [ ]:
# import os
# # os.environ["HOME"] = "~/repos/cobrakbase"
# import cobrakbase

# with open("/home/afreiburger/.kbase/token", 'r') as token:
#     kbase_obj = cobrakbase.KBaseAPI(token.readline())

# media_ws = 207617
# media = kbase_obj.get_from_ws("Wolfe",207617)
2
# from modelseedpy.core.fbahelper import FBAHelper
# from json import dump
# media_js = FBAHelper.convert_kbase_media(media)

# dump(media_js, open("model_inputs/Wolfe.json", 'w'), indent=2)

from json import load
media = load(open("model_inputs/Wolfe.json", 'r'))

### 

### Adding the media and element constraints

In [ ]:
from mscommunity import MSCommunity
from tqdm import tqdm

for ID, model in tqdm(models.items()):
    sample = ID.split("_")[0]
    models[ID] = MSCommunity(model, climit=data_df[sample]["carbon_in"]*1000, o2limit=data_df[sample]["media_cpd00007"]*1000)
    models[ID].util.add_medium = media

## Objective